In [9]:
import os
import pandas as pd

In [10]:
df = pd.read_csv("/Users/hinahaq/Downloads/data_file_project5/final_test_data.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (317235, 7)


,client_id,visitor_id,visit_id,process_step,date_time,source,group_test
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test


## KPI 1: Completion Rate 

**Definition:** % of unique visits reaching `confirm` from `start`  
**Success criteria:** Test > Control by 5%  
**Why:** More completions = more revenue 

In [11]:
completion_rate = (
    df[df['process_step']=='confirm']['visit_id'].nunique() / 
    df[df['process_step']=='start']['visit_id'].nunique() * 100
)
print(f"Completion Rate: {completion_rate:.2f}%")

Completion Rate: 58.92%


In [12]:
completion_by_group = (
    df[df['process_step'] == 'confirm']
    .groupby('group_test')['visit_id'].nunique()
    / df[df['process_step'] == 'start'].groupby('group_test')['visit_id'].nunique()
    * 100
)

print(completion_by_group)

group_test
control    51.912003
test       65.539705
Name: visit_id, dtype: float64


## Task 1 Results 

**Test:** 65.54% completion  
**Control:** 51.91% completion  
**Improvement:** +13.63% (beats 5% threshold!)  

**Learned:** Test group design significantly increases conversions.

## KPI 4: Funnel Drop-off Rate

**Definition:** % users lost between consecutive steps  
**Success:** Test has flatter drop-off curve  
**Why:** Pinpoints problem steps

In [15]:
# Unique visits per step/group
funnel_steps = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
funnel_data = df[df['process_step'].isin(funnel_steps)]

pivot_counts = funnel_data.groupby(['group_test', 'process_step'])['visit_id'].nunique().unstack(fill_value=0)

# Drop-off % between steps
pivot_dropoff = pivot_counts.pct_change(axis=1) * 100
print("Drop-off % by step:")
print(pivot_dropoff.round(1))

Drop-off % by step:
process_step  confirm  start  step_1  step_2  step_3
group_test                                          
control           NaN   92.6   -23.8   -14.5    -9.1
test              NaN   52.6   -14.7   -13.4    -9.5


## Task 4 Results

**Test beats Control everywhere:**
- start → step_1: Test 52.6% drop vs Control 92.6%  
- Consistent lower drops through funnel  

**Learned:** Test design prevents massive early drop-off (92% → 52%).

## KPI 5: Average Steps Reached

**Definition:** Mean furthest step per visit  
**Success:** Test > Control  
**Why:** Proxy for engagement

In [16]:
step_map = {'start':1, 'step_1':2, 'step_2':3, 'step_3':4, 'confirm':5}
df['step_num'] = df['process_step'].map(step_map)

max_steps = df.groupby(['visit_id', 'group_test'])['step_num'].max()
avg_steps = max_steps.groupby('group_test').mean()

print("Avg furthest step reached:")
print(avg_steps.round(2))

Avg furthest step reached:
group_test
control    3.51
test       3.90
Name: step_num, dtype: float64


## Task 5 Results

**Test:** 3.90 steps (reaches step_3)  
**Control:** 3.51 steps (barely reaches step_3)  
**Improvement:** +0.39 steps  

**Learned:** Test users progress 11% further through funnel.